# Bronze to Silver ETL Pipeline

## Objective

This notebook transforms the Bronze layer into the Silver layer by performing data cleaning, schema standardization, feature engineering, validation, and joining product metadata with customer reviews.

The Silver layer acts as the Single Source of Truth (SSOT) for downstream analytics, semantic search, and machine learning use cases.

In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_colwidth", 200)

In [4]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Sports_EDA")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.conf.set("spark.sql.caseSensitive", "true")

In [5]:
PROJECT_ROOT = r"C:\Users\affaa\OneDrive\Desktop\CDAC_PROJECT"

BRONZE_PATH = os.path.join(PROJECT_ROOT, "Bronze")
SILVER_PATH = os.path.join(PROJECT_ROOT, "Silver")

CATEGORY = "Sports_and_Outdoors"
META_CATEGORY = "meta_Sports_and_Outdoors"

In [6]:
# Bronze Reviews Path
reviews_path = os.path.join(
    BRONZE_PATH,
    "reviews",
    CATEGORY
)

# Bronze Metadata Path
metadata_path = os.path.join(
    BRONZE_PATH,
    "metadata",
    META_CATEGORY
)

# Read Bronze Parquet Files
reviews_df = spark.read.parquet(reviews_path)
metadata_df = spark.read.parquet(metadata_path)

print("Bronze datasets loaded successfully.")

Bronze datasets loaded successfully.


print("=" * 80)
print("REVIEWS COLUMNS")
print("=" * 80)

for col in reviews_df.columns:
    print(col)

print("\n" + "=" * 80)
print("METADATA COLUMNS")
print("=" * 80)

for col in metadata_df.columns:
    print(col)

# 6. Silver Layer Transformations

The Silver layer represents the cleaned, standardized, and analytics-ready version of the raw Bronze datasets.

During this stage, raw data is transformed into a structured format by applying data quality checks, standardizing data types, handling missing values where appropriate, and preparing the data for downstream analytical and machine learning workloads.

The objective is to preserve the original business information while improving data consistency, usability, and reliability.

## 6.1 Transformation Strategy

The following transformations will be applied during the Silver layer:

- Remove unnecessary columns where appropriate.
- Standardize column data types.
- Convert timestamps into readable date and time formats.
- Clean and standardize textual fields.
- Handle missing values based on business requirements.
- Flatten or simplify selected nested structures.
- Preserve the product identifier (`parent_asin`) for joining product metadata with customer reviews.
- Validate the transformed datasets before storing them in Parquet format.

## 6.2 Creating Working DataFrames

To preserve the integrity of the Bronze layer, separate working DataFrames are created for the Silver transformation process.

All subsequent cleaning and transformation steps will be applied to these working DataFrames, ensuring that the original Bronze datasets remain unchanged throughout the pipeline.

In [18]:
silver_reviews_df = reviews_df

silver_metadata_df = metadata_df

## 6.3 Converting Review Timestamp

The review timestamp is stored as a Unix epoch timestamp in milliseconds, which is not easily interpretable for analysis.

To improve readability and support time-based analytics, a new timestamp column is created by converting the Unix timestamp into Spark's `TimestampType`.

The original timestamp column is retained to preserve the raw source data for traceability and auditing purposes.

In [19]:
from pyspark.sql import functions as F

silver_reviews_df = silver_reviews_df.withColumn(
    "review_timestamp",
    F.to_timestamp(
        F.from_unixtime(F.col("timestamp") / 1000)
    )
)

In [20]:
silver_reviews_df.select(
    "timestamp",
    "review_timestamp"
).limit(5).toPandas()

,timestamp,review_timestamp
0,1616132396916,2021-03-19 11:09:56
1,1612417884731,2021-02-04 11:21:24
2,1612417377014,2021-02-04 11:12:57
3,1612416926927,2021-02-04 11:05:26
4,1612416614001,2021-02-04 11:00:14


## 6.4 Creating Date-Based Features

To support time-based analysis and reporting, additional date-related columns are derived from the converted review timestamp.

These columns simplify filtering, aggregation, partitioning, and trend analysis without requiring repeated timestamp transformations during downstream processing.

The following attributes are created:

- **review_date**: Date on which the review was posted.
- **review_year**: Year of the review.
- **review_month**: Month of the review.

In [21]:
silver_reviews_df = (
    silver_reviews_df
    .withColumn("review_date", F.to_date("review_timestamp"))
    .withColumn("review_year", F.year("review_timestamp"))
    .withColumn("review_month", F.month("review_timestamp"))
)

In [22]:
silver_reviews_df.select(
    "review_timestamp",
    "review_date",
    "review_year",
    "review_month"
).limit(5).toPandas()

,review_timestamp,review_date,review_year,review_month
0,2021-03-19 11:09:56,2021-03-19,2021,3
1,2021-02-04 11:21:24,2021-02-04,2021,2
2,2021-02-04 11:12:57,2021-02-04,2021,2
3,2021-02-04 11:05:26,2021-02-04,2021,2
4,2021-02-04 11:00:14,2021-02-04,2021,2


## 6.5 Standardizing Text Fields

Textual columns may contain leading or trailing whitespace or empty string values that reduce data quality and complicate downstream processing.

To improve consistency, the review title and review text are standardized by:

- Removing leading and trailing whitespace.
- Converting empty or whitespace-only strings to `NULL`.

This ensures that missing textual information is represented consistently across the dataset.

In [23]:
silver_reviews_df.select(
    "title",
    "text"
).limit(5).toPandas()

,title,text
0,Pretty neat,Eyes glow well. No issues.
1,Pretty good day pack,"I like it for day hikes. Not too big, not too small. Comfortable. No belt, which is what I wanted as I wear a battle belt with this."
2,Small and light,Boils water fast. Fits easily in a med sized MSR pot.
3,Great cold weather stove,"When it gets too cold for butane, this is my go to stove. Lightweight, easy to use, dirt cheap. Not as high speed/low drag as a butane stove but it boils water long after my MSR PocketRocket dies ..."
4,A step above the plastic handle types,A lot more stout than the others I have. Has found a home in my main hiking pack. Worth it.


## 6.6 Renaming and Removing Unnecessary Columns

To improve schema clarity and maintain consistency across the Silver layer, ambiguous column names are renamed to better reflect their business meaning.

Additionally, columns that are either redundant or outside the scope of this project are removed to reduce storage overhead and simplify downstream analytics.

### Renamed Columns

- `rating` → `review_rating`
- `title` → `review_title`
- `text` → `review_text`

### Removed Columns

- `asin` (variant-level identifier; `parent_asin` is used as the common product identifier)
- `images` (sparsely populated and not required for the project)

These changes produce a cleaner and more self-explanatory schema while preserving all information required for analysis.

In [24]:
silver_reviews_df = (
    silver_reviews_df
    .drop("asin", "images")
    .withColumnRenamed("rating", "review_rating")
    .withColumnRenamed("title", "review_title")
    .withColumnRenamed("text", "review_text")
)

In [25]:
print("=" * 80)
print("UPDATED REVIEWS SCHEMA")
print("=" * 80)

silver_reviews_df.printSchema()

UPDATED REVIEWS SCHEMA
root
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- review_title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)



### Reordering Columns

The columns are reordered to improve readability and maintain a logical schema organization.

Business identifiers are placed first, followed by review attributes, analytical date fields, and finally the original raw timestamp retained for data lineage and traceability.

In [26]:
silver_reviews_df = silver_reviews_df.select(
    "parent_asin",
    "user_id",
    "review_rating",
    "review_title",
    "review_text",
    "helpful_vote",
    "verified_purchase",
    "review_timestamp",
    "review_date",
    "review_year",
    "review_month",
    "timestamp"
)

In [27]:
silver_reviews_df.limit(5).toPandas()

,parent_asin,user_id,review_rating,review_title,review_text,helpful_vote,verified_purchase,review_timestamp,review_date,review_year,review_month,timestamp
0,B07HJ2421X,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Pretty neat,Eyes glow well. No issues.,0,True,2021-03-19 11:09:56,2021-03-19,2021,3,1616132396916
1,B0131SE0KI,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Pretty good day pack,"I like it for day hikes. Not too big, not too small. Comfortable. No belt, which is what I wanted as I wear a battle belt with this.",0,True,2021-02-04 11:21:24,2021-02-04,2021,2,1612417884731
2,B01N5O7551,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Small and light,Boils water fast. Fits easily in a med sized MSR pot.,0,True,2021-02-04 11:12:57,2021-02-04,2021,2,1612417377014
3,B01B3NRAUA,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Great cold weather stove,"When it gets too cold for butane, this is my go to stove. Lightweight, easy to use, dirt cheap. Not as high speed/low drag as a butane stove but it boils water long after my MSR PocketRocket dies ...",0,True,2021-02-04 11:05:26,2021-02-04,2021,2,1612416926927
4,B07P5JZ4SN,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,A step above the plastic handle types,A lot more stout than the others I have. Has found a home in my main hiking pack. Worth it.,2,True,2021-02-04 11:00:14,2021-02-04,2021,2,1612416614001


In [28]:
silver_reviews_df = silver_reviews_df.drop("timestamp")

## 6.7 Validating the Reviews Silver Dataset

After applying the Silver layer transformations, the dataset is validated to ensure that the transformations have been successfully applied without affecting the integrity of the data.

The validation includes:

- Total number of records
- Total number of columns
- Updated schema
- Sample records

In [29]:
print("=" * 80)
print("REVIEWS SILVER DATASET SUMMARY")
print("=" * 80)

print(f"Rows    : {silver_reviews_df.count():,}")
print(f"Columns : {len(silver_reviews_df.columns)}")

REVIEWS SILVER DATASET SUMMARY
Rows    : 19,595,170
Columns : 11


In [30]:
silver_reviews_df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)



In [31]:
silver_reviews_df.limit(5).toPandas()

,parent_asin,user_id,review_rating,review_title,review_text,helpful_vote,verified_purchase,review_timestamp,review_date,review_year,review_month
0,B07HJ2421X,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Pretty neat,Eyes glow well. No issues.,0,True,2021-03-19 11:09:56,2021-03-19,2021,3
1,B0131SE0KI,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Pretty good day pack,"I like it for day hikes. Not too big, not too small. Comfortable. No belt, which is what I wanted as I wear a battle belt with this.",0,True,2021-02-04 11:21:24,2021-02-04,2021,2
2,B01N5O7551,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Small and light,Boils water fast. Fits easily in a med sized MSR pot.,0,True,2021-02-04 11:12:57,2021-02-04,2021,2
3,B01B3NRAUA,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,Great cold weather stove,"When it gets too cold for butane, this is my go to stove. Lightweight, easy to use, dirt cheap. Not as high speed/low drag as a butane stove but it boils water long after my MSR PocketRocket dies ...",0,True,2021-02-04 11:05:26,2021-02-04,2021,2
4,B07P5JZ4SN,AFUF6HJRQWUIGBTQNGVZYZJUW6SA,5.0,A step above the plastic handle types,A lot more stout than the others I have. Has found a home in my main hiking pack. Worth it.,2,True,2021-02-04 11:00:14,2021-02-04,2021,2


# 7. Metadata Silver Layer Transformations

The metadata dataset contains product-level information extracted from Amazon's semi-structured JSON data.

Compared to the reviews dataset, the metadata dataset includes several nested structures, arrays, and optional attributes that require additional processing before the data becomes suitable for analytics.

The objective of the Silver layer is to standardize the schema, simplify complex structures, remove unnecessary attributes, and prepare the dataset for downstream analytical workloads while preserving the essential product information.

## 7.1 Creating the Metadata Silver DataFrame

A separate working DataFrame is created to preserve the original Bronze dataset.

All subsequent cleaning and transformation operations will be performed on this working copy while leaving the Bronze layer unchanged.

## 7.2 Metadata Column Assessment

The following table defines the planned transformations for each metadata attribute before implementing the Silver layer.

| Column | Planned Action | Reason |
|----------|----------------|--------|
| parent_asin | Keep | Primary product identifier used for joining with reviews |
| title | Rename | Clarify that this is the product title |
| average_rating | Rename | Differentiate from customer review rating |
| rating_number | Rename | Improve readability |
| categories | Transform | Flatten array into a readable string |
| description | Transform | Flatten array into a readable string |
| features | Transform | Flatten array into a readable string |
| details | Transform | Extract useful information from the nested structure |
| price | Clean | Standardize data type and missing values |
| store | Clean | Standardize text values |
| main_category | Keep | Product categorization |
| author | Evaluate | Retain if meaningful after inspection |
| subtitle | Evaluate | Remove if mostly null |
| bought_together | Remove | Not required for this project |
| images | Remove | Large nested structure not required |
| videos | Remove | Not required for downstream analytics |

## 7.3 Renaming Metadata Columns

To improve schema readability and maintain consistent naming conventions across the Silver layer, several metadata columns are renamed to better reflect their business meaning.

The new names distinguish product-level attributes from review-level attributes and make the dataset more intuitive for downstream analytics.

In [32]:
silver_metadata_df = (
    silver_metadata_df
    .withColumnRenamed("title", "product_title")
    .withColumnRenamed("average_rating", "product_average_rating")
    .withColumnRenamed("rating_number", "product_rating_count")
    .withColumnRenamed("description", "description_text")
    .withColumnRenamed("features", "features_text")
)

In [33]:
silver_metadata_df.printSchema()

root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- product_average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- details: struct (nullable = true)
 |    |-- ABPA Partslink Number: string (nullable = true)
 |    |-- AC Adapter Current: string (nullable = true)
 |    |-- ASTM Fluid Rating: string (nullable = true)
 |    |-- Access Location: string (nullable = true)
 |    |-- Accessory Connection Type: string (nullable = true)
 |    |-- Action: string (nullable = true)
 |    |-- Active Ingredients: string (nullable = true)
 |    |-- Actors: string (nullable = true)
 |    |-- Actuator Type: string (

In [34]:
silver_metadata_df = silver_metadata_df.drop(
    "author",
    "bought_together",
    "subtitle",
    "videos"
)

In [35]:
silver_metadata_df = (
    silver_metadata_df.withColumnRenamed("price", "product_price")
)

In [37]:
silver_metadata_df.printSchema(level=1)

root
 |-- product_average_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |-- description_text: array (nullable = true)
 |-- details: struct (nullable = true)
 |-- features_text: array (nullable = true)
 |-- images: array (nullable = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- product_price: string (nullable = true)
 |-- product_rating_count: long (nullable = true)
 |-- store: string (nullable = true)
 |-- product_title: string (nullable = true)



In [38]:
silver_metadata_df = silver_metadata_df.drop("details")

In [39]:
silver_metadata_df.printSchema()

root
 |-- product_average_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- features_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- hi_res: string (nullable = true)
 |    |    |-- large: string (nullable = true)
 |    |    |-- thumb: string (nullable = true)
 |    |    |-- variant: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- product_price: string (nullable = true)
 |-- product_rating_count: long (nullable = true)
 |-- store: string (nullable = true)
 |-- product_title: string (nullable = true)



## 7.5 Standardizing Product Price

The `product_price` column is stored as a string in the Bronze dataset. To support numerical analysis, aggregation, and reporting, the column is converted into a numeric data type.

Before conversion, currency symbols and other non-numeric characters are removed while preserving missing values as `NULL`.

Missing prices are intentionally retained as null values because the absence of a price does not imply that the product is free.

In [42]:
from pyspark.sql.types import DecimalType

silver_metadata_df = (
    silver_metadata_df
    .withColumn(
        "product_price",
        F.regexp_replace("product_price", r"[^0-9.]", "")
    )
    .withColumn(
        "product_price",
        F.col("product_price").cast(DecimalType(10, 2))
    )
)

In [43]:
silver_metadata_df.select(
    "product_price"
).limit(10).toPandas()

,product_price
0,None
1,None
2,None
3,None
4,None
5,None
6,None
7,None
8,None
9,22.37


In [44]:
silver_metadata_df.filter(
    F.col("product_price").isNotNull()
).select("product_price").show(10, truncate=False)

+-------------+
|product_price|
+-------------+
|22.37        |
|56.99        |
|214.99       |
|69.83        |
|12.96        |
|8.99         |
|14.80        |
|199.95       |
|22.88        |
|14.99        |
+-------------+
only showing top 10 rows


In [45]:
silver_metadata_df.filter(
    F.col("product_price").isNull()
).count()

1102832

In [46]:
silver_metadata_df.select("product_price").show(10)

+-------------+
|product_price|
+-------------+
|         NULL|
|         NULL|
|         NULL|
|         NULL|
|         NULL|
|         NULL|
|         NULL|
|         NULL|
|         NULL|
|        22.37|
+-------------+
only showing top 10 rows


In [47]:
silver_metadata_df.select("images").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 7.6 Extracting the Primary Product Image

The original `images` column contains an array of image objects with multiple resolutions and variants for each product.

For this project, a single representative product image is sufficient for downstream analytics, dashboards, and application development.

The transformation extracts the highest-quality available image by selecting the first image in the array and prioritizing the `hi_res` image. If a high-resolution image is unavailable, the `large` image is used as a fallback.

After extracting the representative image, the original nested `images` column is removed to simplify the schema.

In [48]:
silver_metadata_df = (
    silver_metadata_df
    .withColumn(
        "product_image_url",
        F.when(
            F.col("images").isNotNull() & (F.size("images") > 0),
            F.coalesce(
                F.col("images")[0]["hi_res"],
                F.col("images")[0]["large"]
            )
        )
    )
    .drop("images")
)

In [49]:
silver_metadata_df.select(
    "product_image_url"
).show(10, truncate=False)

+---------------------------------------------------------------+
|product_image_url                                              |
+---------------------------------------------------------------+
|https://m.media-amazon.com/images/I/11sRmEBaZvL._AC_.jpg       |
|https://m.media-amazon.com/images/I/71RDUlyuzaL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/41UVNs6PA6L._AC_.jpg       |
|https://m.media-amazon.com/images/I/71QCHI-STOL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/A1yxMGigPGL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/61IuLpRAGDL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/61WpCfiAYwL._AC_SL1017_.jpg|
|https://m.media-amazon.com/images/I/71tKZo1NtEL._AC_SL1500_.jpg|
|https://m.media-amazon.com/images/I/51uDJvEaAEL._AC_.jpg       |
|https://m.media-amazon.com/images/I/618GCgG5FEL._AC_SL1001_.jpg|
+---------------------------------------------------------------+
only showing top 10 rows


In [50]:
silver_metadata_df.select("product_image_url").show(10, truncate=False)

+---------------------------------------------------------------+
|product_image_url                                              |
+---------------------------------------------------------------+
|https://m.media-amazon.com/images/I/11sRmEBaZvL._AC_.jpg       |
|https://m.media-amazon.com/images/I/71RDUlyuzaL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/41UVNs6PA6L._AC_.jpg       |
|https://m.media-amazon.com/images/I/71QCHI-STOL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/A1yxMGigPGL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/61IuLpRAGDL._AC_UL1500_.jpg|
|https://m.media-amazon.com/images/I/61WpCfiAYwL._AC_SL1017_.jpg|
|https://m.media-amazon.com/images/I/71tKZo1NtEL._AC_SL1500_.jpg|
|https://m.media-amazon.com/images/I/51uDJvEaAEL._AC_.jpg       |
|https://m.media-amazon.com/images/I/618GCgG5FEL._AC_SL1001_.jpg|
+---------------------------------------------------------------+
only showing top 10 rows


In [51]:
silver_metadata_df.select("product_image_url").limit(1).collect()[0][0]

'https://m.media-amazon.com/images/I/11sRmEBaZvL._AC_.jpg'

In [52]:
silver_metadata_df.select("product_image_url").toPandas()

,product_image_url
0,https://m.media-amazon.com/images/I/11sRmEBaZvL._AC_.jpg
1,https://m.media-amazon.com/images/I/71RDUlyuzaL._AC_UL1500_.jpg
2,https://m.media-amazon.com/images/I/41UVNs6PA6L._AC_.jpg
3,https://m.media-amazon.com/images/I/71QCHI-STOL._AC_UL1500_.jpg
4,https://m.media-amazon.com/images/I/A1yxMGigPGL._AC_UL1500_.jpg
...,...
1587416,https://m.media-amazon.com/images/I/61dzRl8jH2L._AC_UL1500_.jpg
1587417,https://m.media-amazon.com/images/I/61jMm-ijwUL._AC_SL1001_.jpg
1587418,https://m.media-amazon.com/images/I/91Llaw5cFVL._AC_SL1500_.jpg
1587419,https://m.media-amazon.com/images/I/61kGS-3jKeL._AC_SL1500_.jpg


In [53]:
silver_metadata_df.printSchema()

root
 |-- product_average_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- features_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_rating_count: long (nullable = true)
 |-- store: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_image_url: string (nullable = true)



In [54]:
silver_metadata_df.filter(
    F.col("product_image_url").isNull()
).count()

540

In [55]:
silver_metadata_df = silver_metadata_df.drop("images")

In [56]:
silver_metadata_df.select(
    "description_text"
).show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|description_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
+-------

In [57]:
silver_metadata_df.filter(
    F.size("description_text") == 0
).count()

553002

In [58]:
silver_metadata_df.filter(
    F.size("description_text") > 1
).count()

164316

## 7.7 Flattening Product Descriptions

The `description_text` column is stored as an array of strings in the Bronze dataset. While this structure preserves the original data, it is not ideal for analytics, reporting, or semantic search applications.

To simplify downstream processing, all description elements are combined into a single text field while preserving their original order. Empty arrays are converted to `NULL` to accurately represent missing descriptions.

This transformation produces a clean textual representation that can be directly used for natural language processing, embedding generation, and product search.

In [59]:
silver_metadata_df = silver_metadata_df.withColumn(
    "description_text",
    F.when(
        F.size("description_text") == 0,
        None
    ).otherwise(
        F.array_join("description_text", " ")
    )
)

In [60]:
silver_metadata_df.select(
    "description_text"
).show(10, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [61]:
silver_metadata_df.filter(
    F.col("description_text").isNull()
).count()

553002

In [62]:
silver_metadata_df.select(
    "features_text"
).show(10, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 7.8 Flattening Product Features

The `features_text` column is stored as an array of strings containing product features, specifications, package contents, and marketing highlights.

To improve usability for downstream analytics and natural language processing, the array is converted into a single text field while preserving all available information. Individual feature entries are separated using the pipe (`|`) delimiter, maintaining a clear distinction between each feature. Empty arrays are converted to `NULL` to accurately represent missing values.

This transformation creates a clean textual representation suitable for reporting, search, and future semantic embedding generation without losing any feature information.

In [63]:
silver_metadata_df = silver_metadata_df.withColumn(
    "features_text",
    F.when(
        F.size("features_text") == 0,
        None
    ).otherwise(
        F.array_join("features_text", " | ")
    )
)

In [66]:
silver_metadata_df.select(
    "features_text"
).limit(10).toPandas()

,features_text
0,Return Elasticity | High Performance | Controls your sweat
1,100% polyester material | Imported | Hand Wash Only | Decorated in full team colors with official logos | V-neck collar | High cut on stomach area for comfort fit for male dogs | Machine washable ...
2,PACKAGE INCLUDES: 6 Pcs.L.A. Lakers Cuff Band Favors | ADORABLE DESIGN: These awesome rubber cuff-bands are the ultimate party accessory to show how much you and your party goers love the L.A. Lak...
3,100% Leather | Imported | Leather upper | Rubber sole | Sipped non-slip outsole | Leather lace | Traditional boat shoe
4,"UA Storm technology repels water without sacrificing breathability | Windproof construction shields you from the elements | Durable, bonded 3 layer softershell material has a quiet outer layer & s..."
5,Imported | Plastic frame | Plastic lens | Polarized | Mirror Coating coating | Lens width: 61 millimeters | Lens height: 44 millimeters | Bridge: 17 millimeters | Arm: 130 millimeters | 【POLARIZED...
6,"✔100% Brand new and high quality. | ✔Clip on your fishing rod easily, With twin bells bite alarm | ✔Great for professional and occasional fishermen, Ensure fishing line is able to be seen in the d..."
7,"✔ MORE SAFE THAN STAINLESS CAMPING FLATWARE: Because of titanium’s excellent bio-compatibility, it’s completely non-toxic, harmless, and non-allergic to the human body.(stainless steel generally c..."
8,"𝗦𝘁𝗮𝘆 𝗛𝗼𝘁 𝗼𝗿 𝗦𝘁𝗮𝘆 𝗖𝗼𝗹𝗱 - Retain the temperature of your chosen beverage for a prolonged period of time when you choose Veli. Any drink that is hot, like coffee, can stay hot for up to 6 hours. Cool..."
9,"High Quality Material:100% high quality material.The metal part of our ycle saddle including two springs and clamp with electroplating processing,which is strong strength and anti-,good shock resi..."


In [67]:
silver_metadata_df.filter(
    F.col("features_text").isNull()
).count()

268577

## 7.9 Validating Product Category Information

The metadata dataset contains two category-related fields:

- `main_category`
- `categories`

Before engineering new category columns, both fields are analyzed to understand their relationship, identify inconsistencies, and determine how each should be used in the Silver layer.

This validation ensures that no useful business information is lost during transformation and that any derived category fields accurately represent the product hierarchy.

In [77]:
silver_metadata_df.select(
    F.count(F.when(F.col("main_category").isNull(), 1)).alias("main_category_nulls"),
    F.count(F.when(F.col("categories").isNull(), 1)).alias("categories_nulls")
).toPandas()

,main_category_nulls,categories_nulls
0,249954,0


In [78]:
silver_metadata_df.filter(
    F.size("categories") == 0
).count()

89830

In [79]:
silver_metadata_df.groupBy(
    "main_category"
).count().orderBy(
    F.desc("count")
).toPandas()

,main_category,count
0,Sports & Outdoors,870267
1,AMAZON FASHION,259564
2,NaN,249954
3,Amazon Home,74995
4,Automotive,45629
5,Tools & Home Improvement,25580
6,Toys & Games,12477
7,Industrial & Scientific,9081
8,Pet Supplies,7999
9,Cell Phones & Accessories,6905


In [80]:
silver_metadata_df.groupBy(
    "categories"
).count().orderBy(
    F.desc("count")
).limit(20).toPandas()

,categories,count
0,[],89830
1,"[Sports & Outdoors, Fan Shop, Clothing, T-Shirts]",49706
2,"[Sports & Outdoors, Fan Shop, Auto Accessories, Decals & Bumper Stickers, Decals]",29189
3,"[Sports & Outdoors, Sports & Outdoor Recreation Accessories, Sports Water Bottles]",28335
4,"[Sports & Outdoors, Fan Shop, Clothing Accessories, Caps & Hats, Baseball Caps]",26544
5,"[Sports & Outdoors, Fan Shop, Clothing, Sweatshirts & Hoodies]",20714
6,"[Sports & Outdoors, Fan Shop, Clothing, Jerseys]",17432
7,"[Sports & Outdoors, Hunting & Fishing, Shooting, Gun Accessories, Maintenance & Storage, Gun Holsters, Cases & Bags, Gun Holsters]",16269
8,"[Sports & Outdoors, Hunting & Fishing, Fishing, Accessories, Fishing Hats]",13839
9,"[Sports & Outdoors, Exercise & Fitness, Clothing]",11408


In [81]:
silver_metadata_df.select(
    "main_category",
    F.get("categories", 0).alias("first_category"),
    F.get("categories", -1).alias("last_category")
).limit(20).toPandas()

,main_category,first_category,last_category
0,NaN,Sports & Outdoors,None
1,NaN,Sports & Outdoors,None
2,Toys & Games,Sports & Outdoors,None
3,NaN,Sports & Outdoors,None
4,NaN,Sports & Outdoors,None
5,NaN,Sports & Outdoors,None
6,NaN,Sports & Outdoors,None
7,NaN,Sports & Outdoors,None
8,NaN,Sports & Outdoors,None
9,Sports & Outdoors,Sports & Outdoors,None


In [82]:
silver_metadata_df.select(
    "categories"
).where(
    F.size("categories") >= 5
).limit(20).toPandas()

,categories
0,"[Sports & Outdoors, Hunting & Fishing, Fishing, Terminal Tackle & Accessories, Dodgers & Flashers]"
1,"[Sports & Outdoors, Outdoor Recreation, Camping & Hiking, Camp Kitchen, Dishes & Utensils, Flatware]"
2,"[Sports & Outdoors, Fan Shop, Home & Kitchen, Kitchen & Dining, Cups & Glasses, Tumblers]"
3,"[Sports & Outdoors, Sports, Cycling, Parts & Components, Seats & Saddles, Saddles]"
4,"[Sports & Outdoors, Sports, Team Sports, Lacrosse, Field Equipment, Goals]"
5,"[Sports & Outdoors, Sports, Cycling, Car Racks, Transportation & Storage, Indoor Bike Storage]"
6,"[Sports & Outdoors, Sports, Winter Sports, Winter Sports Accessories, Goggles & Lenses, Goggles]"
7,"[Sports & Outdoors, Sports, Golf, Golf Clubs, Irons, Single Clubs]"
8,"[Sports & Outdoors, Sports, Team Sports, Softball, Bats, Fast-Pitch Softball Bats]"
9,"[Sports & Outdoors, Fan Shop, Clothing Accessories, Caps & Hats, Baseball Caps]"


In [84]:
from pyspark.sql import functions as F

silver_metadata_df.select(
    "main_category",
    F.try_element_at("categories", F.lit(1)).alias("first_category"),
    F.try_element_at("categories", F.lit(-1)).alias("last_category"),
    F.concat_ws(" | ", "categories").alias("category_path")
).limit(30).toPandas()

,main_category,first_category,last_category,category_path
0,NaN,Sports & Outdoors,Resistance Bands,Sports & Outdoors | Exercise & Fitness | Strength Training Equipment | Resistance Bands
1,NaN,Sports & Outdoors,Jerseys,Sports & Outdoors | Fan Shop | Clothing | Jerseys
2,Toys & Games,Sports & Outdoors,Resistance Bands,Sports & Outdoors | Exercise & Fitness | Strength Training Equipment | Resistance Bands
3,NaN,Sports & Outdoors,Boating,Sports & Outdoors | Sports | Boating & Sailing | Boating
4,NaN,Sports & Outdoors,Under Armour Deals,Sports & Outdoors | Under Armour Deals
5,NaN,Sports & Outdoors,Sunglasses,Sports & Outdoors | Fan Shop | Clothing Accessories | Sunglasses
6,NaN,Sports & Outdoors,Dodgers & Flashers,Sports & Outdoors | Hunting & Fishing | Fishing | Terminal Tackle & Accessories | Dodgers & Flashers
7,NaN,Sports & Outdoors,Flatware,Sports & Outdoors | Outdoor Recreation | Camping & Hiking | Camp Kitchen | Dishes & Utensils | Flatware
8,NaN,Sports & Outdoors,Tumblers,Sports & Outdoors | Fan Shop | Home & Kitchen | Kitchen & Dining | Cups & Glasses | Tumblers
9,Sports & Outdoors,Sports & Outdoors,Saddles,Sports & Outdoors | Sports | Cycling | Parts & Components | Seats & Saddles | Saddles


In [85]:
from pyspark.sql import functions as F

silver_metadata_df.select(
    "main_category",
    F.when(
        F.size("categories") > 0,
        F.col("categories")[0]
    ).alias("first_category"),
    F.when(
        F.size("categories") > 0,
        F.expr("categories[size(categories)-1]")
    ).alias("last_category"),
    F.concat_ws(" | ", "categories").alias("category_path")
).limit(30).toPandas()

,main_category,first_category,last_category,category_path
0,NaN,Sports & Outdoors,Resistance Bands,Sports & Outdoors | Exercise & Fitness | Strength Training Equipment | Resistance Bands
1,NaN,Sports & Outdoors,Jerseys,Sports & Outdoors | Fan Shop | Clothing | Jerseys
2,Toys & Games,Sports & Outdoors,Resistance Bands,Sports & Outdoors | Exercise & Fitness | Strength Training Equipment | Resistance Bands
3,NaN,Sports & Outdoors,Boating,Sports & Outdoors | Sports | Boating & Sailing | Boating
4,NaN,Sports & Outdoors,Under Armour Deals,Sports & Outdoors | Under Armour Deals
5,NaN,Sports & Outdoors,Sunglasses,Sports & Outdoors | Fan Shop | Clothing Accessories | Sunglasses
6,NaN,Sports & Outdoors,Dodgers & Flashers,Sports & Outdoors | Hunting & Fishing | Fishing | Terminal Tackle & Accessories | Dodgers & Flashers
7,NaN,Sports & Outdoors,Flatware,Sports & Outdoors | Outdoor Recreation | Camping & Hiking | Camp Kitchen | Dishes & Utensils | Flatware
8,NaN,Sports & Outdoors,Tumblers,Sports & Outdoors | Fan Shop | Home & Kitchen | Kitchen & Dining | Cups & Glasses | Tumblers
9,Sports & Outdoors,Sports & Outdoors,Saddles,Sports & Outdoors | Sports | Cycling | Parts & Components | Seats & Saddles | Saddles


## 7.10 Standardizing Product Category Information

The original metadata contains two category-related columns:

- `main_category`
- `categories`

Data validation revealed that the `main_category` column contains a large number of missing values and does not consistently represent the actual product taxonomy. In many cases, it contains marketplace-specific classifications that differ from the hierarchical category information stored in `categories`.

The `categories` column, however, provides a complete hierarchical classification for each product and serves as a more reliable source for category information.

To create a standardized and business-friendly schema, the original `main_category` column is removed. The first element of the `categories` array is extracted as the new `main_category`, representing the primary product category, while the remaining hierarchy levels are concatenated into a new `sub_category` column using the pipe (`|`) separator.

This approach preserves the complete category hierarchy in a readable format, eliminates inconsistent metadata, and provides a clean structure for downstream analytics and semantic search applications.

In [86]:
from pyspark.sql import functions as F

silver_metadata_df = silver_metadata_df.drop("main_category")

In [87]:
silver_metadata_df = silver_metadata_df.withColumn(
    "main_category",
    F.when(
        F.size("categories") > 0,
        F.col("categories")[0]
    )
)

In [88]:
silver_metadata_df.select(
    "main_category",
    "categories"
).limit(10).toPandas()

,main_category,categories
0,Sports & Outdoors,"[Sports & Outdoors, Exercise & Fitness, Strength Training Equipment, Resistance Bands]"
1,Sports & Outdoors,"[Sports & Outdoors, Fan Shop, Clothing, Jerseys]"
2,Sports & Outdoors,"[Sports & Outdoors, Exercise & Fitness, Strength Training Equipment, Resistance Bands]"
3,Sports & Outdoors,"[Sports & Outdoors, Sports, Boating & Sailing, Boating]"
4,Sports & Outdoors,"[Sports & Outdoors, Under Armour Deals]"
5,Sports & Outdoors,"[Sports & Outdoors, Fan Shop, Clothing Accessories, Sunglasses]"
6,Sports & Outdoors,"[Sports & Outdoors, Hunting & Fishing, Fishing, Terminal Tackle & Accessories, Dodgers & Flashers]"
7,Sports & Outdoors,"[Sports & Outdoors, Outdoor Recreation, Camping & Hiking, Camp Kitchen, Dishes & Utensils, Flatware]"
8,Sports & Outdoors,"[Sports & Outdoors, Fan Shop, Home & Kitchen, Kitchen & Dining, Cups & Glasses, Tumblers]"
9,Sports & Outdoors,"[Sports & Outdoors, Sports, Cycling, Parts & Components, Seats & Saddles, Saddles]"


### Creating the `sub_category` Column

The first element of the category hierarchy has been extracted as the standardized `main_category`. The remaining hierarchy levels are combined into a single `sub_category` column using the pipe (`|`) separator.

This transformation preserves the complete hierarchical classification while avoiding duplication of the main category. Products with only a single category level or an empty category hierarchy will have a `NULL` value for `sub_category`.

In [89]:
from pyspark.sql import functions as F

silver_metadata_df = silver_metadata_df.withColumn(
    "sub_category",
    F.when(
        F.size("categories") > 1,
        F.concat_ws(
            " | ",
            F.expr("slice(categories, 2, size(categories) - 1)")
        )
    )
)

In [90]:
silver_metadata_df.select(
    "main_category",
    "sub_category",
    "categories"
).limit(10).toPandas()

,main_category,sub_category,categories
0,Sports & Outdoors,Exercise & Fitness | Strength Training Equipment | Resistance Bands,"[Sports & Outdoors, Exercise & Fitness, Strength Training Equipment, Resistance Bands]"
1,Sports & Outdoors,Fan Shop | Clothing | Jerseys,"[Sports & Outdoors, Fan Shop, Clothing, Jerseys]"
2,Sports & Outdoors,Exercise & Fitness | Strength Training Equipment | Resistance Bands,"[Sports & Outdoors, Exercise & Fitness, Strength Training Equipment, Resistance Bands]"
3,Sports & Outdoors,Sports | Boating & Sailing | Boating,"[Sports & Outdoors, Sports, Boating & Sailing, Boating]"
4,Sports & Outdoors,Under Armour Deals,"[Sports & Outdoors, Under Armour Deals]"
5,Sports & Outdoors,Fan Shop | Clothing Accessories | Sunglasses,"[Sports & Outdoors, Fan Shop, Clothing Accessories, Sunglasses]"
6,Sports & Outdoors,Hunting & Fishing | Fishing | Terminal Tackle & Accessories | Dodgers & Flashers,"[Sports & Outdoors, Hunting & Fishing, Fishing, Terminal Tackle & Accessories, Dodgers & Flashers]"
7,Sports & Outdoors,Outdoor Recreation | Camping & Hiking | Camp Kitchen | Dishes & Utensils | Flatware,"[Sports & Outdoors, Outdoor Recreation, Camping & Hiking, Camp Kitchen, Dishes & Utensils, Flatware]"
8,Sports & Outdoors,Fan Shop | Home & Kitchen | Kitchen & Dining | Cups & Glasses | Tumblers,"[Sports & Outdoors, Fan Shop, Home & Kitchen, Kitchen & Dining, Cups & Glasses, Tumblers]"
9,Sports & Outdoors,Sports | Cycling | Parts & Components | Seats & Saddles | Saddles,"[Sports & Outdoors, Sports, Cycling, Parts & Components, Seats & Saddles, Saddles]"


In [91]:
silver_metadata_df = silver_metadata_df.drop("categories")

In [92]:
silver_metadata_df.select(
    "main_category",
    "sub_category"
).limit(10).toPandas()

,main_category,sub_category
0,Sports & Outdoors,Exercise & Fitness | Strength Training Equipment | Resistance Bands
1,Sports & Outdoors,Fan Shop | Clothing | Jerseys
2,Sports & Outdoors,Exercise & Fitness | Strength Training Equipment | Resistance Bands
3,Sports & Outdoors,Sports | Boating & Sailing | Boating
4,Sports & Outdoors,Under Armour Deals
5,Sports & Outdoors,Fan Shop | Clothing Accessories | Sunglasses
6,Sports & Outdoors,Hunting & Fishing | Fishing | Terminal Tackle & Accessories | Dodgers & Flashers
7,Sports & Outdoors,Outdoor Recreation | Camping & Hiking | Camp Kitchen | Dishes & Utensils | Flatware
8,Sports & Outdoors,Fan Shop | Home & Kitchen | Kitchen & Dining | Cups & Glasses | Tumblers
9,Sports & Outdoors,Sports | Cycling | Parts & Components | Seats & Saddles | Saddles


In [93]:
silver_metadata_df.select(
    F.count(F.when(F.col("store").isNull(), 1)).alias("nulls"),
    F.count(F.when(F.trim(F.col("store")) == "", 1)).alias("empty_strings"),
    F.countDistinct("store").alias("distinct_stores")
).toPandas()

,nulls,empty_strings,distinct_stores
0,35109,0,164491


In [94]:
silver_metadata_df.groupBy("store") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(30) \
    .toPandas()

,store,count
0,NaN,35109
1,adidas,16327
2,WinCraft,15917
3,New Era,11442
4,'47,10858
5,Outerstuff,10026
6,Nike,8791
7,Reebok,7986
8,Majestic,7772
9,FOCO,7551


In [95]:
from pyspark.sql import functions as F

silver_metadata_df.groupBy(
    F.upper("store").alias("normalized_store")
).count().orderBy(
    F.desc("count")
).limit(50).toPandas()

,normalized_store,count
0,NaN,35109
1,ADIDAS,16332
2,WINCRAFT,15943
3,NEW ERA,11444
4,'47,10858
5,MAJESTIC,10766
6,OUTERSTUFF,10028
7,NIKE,8803
8,REEBOK,7986
9,FOCO,7580


In [96]:
from pyspark.sql import functions as F

silver_metadata_df.filter(F.col("store").isNotNull()) \
    .groupBy(F.upper(F.col("store")).alias("normalized_store")) \
    .agg(
        F.collect_set("store").alias("store_variants"),
        F.count("*").alias("total_records")
    ) \
    .filter(F.size("store_variants") > 1) \
    .orderBy(F.desc("total_records")) \
    .toPandas()

,normalized_store,store_variants,total_records
0,ADIDAS,"[Adidas, adidas]",16332
1,WINCRAFT,"[Wincraft, WinCraft, wincraft]",15943
2,NEW ERA,"[NEW ERA, New Era]",11444
3,MAJESTIC,"[Majestic, MAJESTIC]",10766
4,OUTERSTUFF,"[Outerstuff, outerstuff, OuterStuff]",10028
...,...,...,...
3888,UNIMER,"[Unimer, UNIMER]",2
3889,USSC PRODUCTS,"[USSC products, USSC Products]",2
3890,WELSPO,"[welspo, WELSPO]",2
3891,WISDOM,"[Wisdom, WISDOM]",2


### Standardizing Store Names

During data profiling, multiple records were found where the same store or brand appeared with different capitalization (e.g., `adidas` vs `Adidas`, `WinCraft` vs `Wincraft` vs `wincraft`, `Majestic` vs `MAJESTIC`).

Applying a generic case transformation (such as converting all values to uppercase or title case) would incorrectly modify official brand names (e.g., `adidas`, `SHIMANO`, `VF LSG`, `'47`).

Instead, a frequency-based standardization approach was used.

For each store name:

- The store name was normalized to uppercase for comparison.
- All capitalization variants were grouped together.
- The most frequently occurring variant was selected as the canonical representation.
- All other variants were replaced with the canonical value.

This approach preserves official brand naming while eliminating inconsistent capitalization across the dataset.

In [97]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Find the most frequent capitalization for each normalized store name
store_mapping = (
    silver_metadata_df
    .filter(F.col("store").isNotNull())
    .groupBy(
        F.upper("store").alias("normalized_store"),
        "store"
    )
    .count()
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("normalized_store")
                  .orderBy(F.desc("count"))
        )
    )
    .filter(F.col("rank") == 1)
    .select(
        "normalized_store",
        F.col("store").alias("canonical_store")
    )
)

# Apply the canonical store names
silver_metadata_df = (
    silver_metadata_df
    .withColumn("normalized_store", F.upper("store"))
    .join(store_mapping, on="normalized_store", how="left")
    .withColumn(
        "store",
        F.when(
            F.col("store").isNull(),
            None
        ).otherwise(F.col("canonical_store"))
    )
    .drop("normalized_store", "canonical_store")
)

In [98]:
silver_metadata_df.groupBy(
    F.upper("store").alias("normalized_store")
).agg(
    F.collect_set("store").alias("store_variants")
).filter(
    F.size("store_variants") > 1
).count()

0

In [99]:
silver_metadata_df.groupBy("store") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(30) \
    .toPandas()

,store,count
0,NaN,35109
1,adidas,16332
2,WinCraft,15943
3,New Era,11444
4,'47,10858
5,Majestic,10766
6,Outerstuff,10028
7,Nike,8803
8,Reebok,7986
9,FOCO,7580


### Validating Product Titles

Product titles are one of the primary descriptive attributes used throughout the analytics pipeline and will later contribute to semantic search.

The validation checks include:

- Missing values
- Empty strings
- Duplicate titles (expected since different products can share the same title)
- Leading and trailing whitespace

Only minimal cleaning is performed to preserve the original product information.

In [100]:
silver_metadata_df.select(
    F.count(F.when(F.col("product_title").isNull(), 1)).alias("nulls"),
    F.count(F.when(F.trim("product_title") == "", 1)).alias("empty_strings"),
    F.countDistinct("product_title").alias("distinct_titles")
).toPandas()

,nulls,empty_strings,distinct_titles
0,0,112,1354655


In [101]:
silver_metadata_df.select("product_title").limit(20).toPandas()

,product_title
0,MaxFlowSports Cross-Grip Hairband
1,"NCAA Florida State Seminoles Pet Jersey, X-Large"
2,"Official L.A. Lakers Cuff Band Favors - 4""x1"" (Pack Of 6) | Purple & Yellow Rubber Wristbands For Sports Fans & Parties"
3,Columbia Men's Perfect Cast Boat Shoe
4,Under Armour Teen-Boy's Storm Softershell Jacket
5,YDAOWKN Fit Over Glasses Sunglasses for Men Women Wrap Around Polarized Driving Sunglasses 100% UV400 Protection
6,Quaanti Hot Sale 10PCS LED Night Fishing Rod Bite Bait Alarm Light Twin Bells Clip Alerter Sea Fishing Practical Tools Lowest Price (Green)
7,KUCHYNEE Titanium Camping Cutlery Set Utility Utensils Flatware Ultra Lightweight 3 Pcs Portable Knife Fork Spoon for Home Outdoor Travel Hiking Camping BBQ Barbecue with Case and Bag
8,Personalized Custom Name Tumblers Cup Taino Indian Puerto Rico Flag Stainless Steel Tumbler With Lid Customized Gifts For Boricuas Friends Unique Presents For Puerto Ricans
9,"Tbest Universal Comfortable ycle Saddle,Brown Leather Bike,Comfortable Rivets Bike Durable PU Leather Spring ycle Saddle with Soft Cushion for Men Women Brown Brown Leather Bike Brown Bike Saddle"


### Standardizing Product Titles

Product titles were validated to identify missing or invalid values.

The validation showed:

- No NULL values.
- A small number of empty string values.

Empty strings do not provide meaningful product information and were standardized to NULL to ensure consistent handling of missing data throughout the analytics pipeline.

In [102]:
silver_metadata_df = silver_metadata_df.withColumn(
    "product_title",
    F.when(
        F.trim(F.col("product_title")) == "",
        None
    ).otherwise(
        F.trim(F.col("product_title"))
    )
)

In [103]:
silver_metadata_df.select(
    F.count(F.when(F.col("product_title").isNull(), 1)).alias("null_titles")
).toPandas()

,null_titles
0,112


In [104]:
silver_metadata_df.select(
    F.count(F.when(F.col("product_image_url").isNull(), 1)).alias("nulls"),
    F.count(F.when(F.trim(F.col("product_image_url")) == "", 1)).alias("empty_strings"),
    F.countDistinct("product_image_url").alias("distinct_urls")
).toPandas()

,nulls,empty_strings,distinct_urls
0,540,0,1403575


### Reordering Columns

The Silver dataset columns were reordered into a logical business-oriented structure.

This improves readability and provides a consistent schema for downstream analytics, joins, and data warehouse operations without affecting the underlying data.

In [105]:
silver_metadata_df = silver_metadata_df.select(
    "parent_asin",
    "product_title",
    "store",
    "main_category",
    "sub_category",
    "product_price",
    "product_average_rating",
    "product_rating_count",
    "description_text",
    "features_text",
    "product_image_url"
)

In [106]:
silver_metadata_df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- store: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_average_rating: double (nullable = true)
 |-- product_rating_count: long (nullable = true)
 |-- description_text: string (nullable = true)
 |-- features_text: string (nullable = true)
 |-- product_image_url: string (nullable = true)



In [107]:
print(silver_metadata_df.columns)

['parent_asin', 'product_title', 'store', 'main_category', 'sub_category', 'product_price', 'product_average_rating', 'product_rating_count', 'description_text', 'features_text', 'product_image_url']


In [108]:
silver_reviews_df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)



In [109]:
print(silver_reviews_df.columns)

['parent_asin', 'user_id', 'review_rating', 'review_title', 'review_text', 'helpful_vote', 'verified_purchase', 'review_timestamp', 'review_date', 'review_year', 'review_month']


In [110]:
print(f"Metadata Rows : {silver_metadata_df.count():,}")
print(f"Metadata Columns : {len(silver_metadata_df.columns)}")

print(f"Reviews Rows : {silver_reviews_df.count():,}")
print(f"Reviews Columns : {len(silver_reviews_df.columns)}")

Metadata Rows : 1,587,421
Metadata Columns : 11
Reviews Rows : 19,595,170
Reviews Columns : 11
